# Fink/LSST — Reload Cepheid/Pulsator Light Curves & Analysis

This notebook reloads the data saved by `02_cepheids_extended_search.ipynb`
from the `data_CEPHEIDS_DDF_02/` directory and reproduces the full analysis
(raw light curves, Lomb-Scargle period search, phase-folded diagrams,
Period-Luminosity diagram) **without any Fink API call**.

### Expected directory layout
```
data_CEPHEIDS_DDF_02/
├── df_obj.parquet
├── df_pulsators.parquet
├── df_periods.parquet        (optional — recomputed below if absent)
├── lc_dict_meta.parquet
└── lightcurves/
    └── {diaObjectId}/
        ├── sources.parquet
        └── fp.parquet
```


- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS, Université Paris-Saclay
- created : 2026-06-17
- last update : 2026-06-17

## 1. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import pathlib
import warnings
import os

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from astropy.timeseries import LombScargle
from astropy.time import Time
from IPython.display import display

warnings.filterwarnings("ignore")
print(f"pandas {pd.__version__}  |  numpy {np.__version__}")

In [ ]:
# to enlarge the sizes
params = {
    "legend.fontsize": "large",
    "figure.figsize": (10, 6),
    "axes.labelsize": "large",
    "axes.titlesize": "large",
    "xtick.labelsize": "large",
    "ytick.labelsize": "large",
}
plt.rcParams.update(params)

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl → %matplotlib widget")
except ImportError:
    %matplotlib inline
    print("no ipympl → %matplotlib inline")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
NB_TAG = "CEPHEIDS_DDF_02"
DIR_DATA = pathlib.Path(f"data_{NB_TAG}")
LC_DIR = DIR_DATA / "lightcurves"

NB_TAG_04 = "CEPHEIDS_DDF_04"
DIR_FIGS = pathlib.Path(f"figs_{NB_TAG_04}")
DIR_FIGS.mkdir(parents=True, exist_ok=True)

assert DIR_DATA.exists(), (
    f"Data directory '{DIR_DATA}' not found.\nRun 02_cepheids_extended_search.ipynb first (section 15)."
)
print(f"Data directory : {DIR_DATA.resolve()}")
print(f"LC directory   : {LC_DIR.resolve()}")
print(f"Figs directory : {DIR_FIGS.resolve()}")

# ── Analysis parameters (must match notebook 02) ──────────────────────────────
SNR_MIN = 3.0
BANDS = list("ugrizy")
PERIOD_MIN_DAYS = 0.3
PERIOD_MAX_DAYS = 200.0
LS_SAMPLES = 20

BAND_COLORS = {"u": "#9b59b6", "g": "#2ecc71", "r": "#e74c3c", "i": "#e67e22", "z": "#3498db", "y": "#795548"}
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.5,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 12,
    }
)


def savefig(name):
    for ext in ("pdf", "png"):
        plt.savefig(DIR_FIGS / f"{name}.{ext}", bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print("Configuration done.")

## 2. Helper functions

In [ ]:
def flux_to_mag(flux, flux_err, zp=31.4):
    """Convert nJy flux → AB magnitude and propagated error."""
    with np.errstate(invalid="ignore", divide="ignore"):
        mag = zp - 2.5 * np.log10(np.where(flux > 0, flux, np.nan))
        mag_err = (2.5 / np.log(10)) * np.abs(flux_err / np.where(flux > 0, flux, np.nan))
    return mag, mag_err


def filter_lc(df: pd.DataFrame, snr_min: float = SNR_MIN) -> pd.DataFrame:
    """Apply SNR cut and add mag/mag_err columns to a sources DataFrame."""
    if df.empty:
        return df
    df = df.copy()
    flux_col = "r:psfFlux"
    err_col = "r:psfFluxErr"
    if flux_col not in df.columns or err_col not in df.columns:
        return df
    snr = np.abs(df[flux_col]) / df[err_col].replace(0, np.nan)
    df = df[snr >= snr_min].copy()
    df["mag"], df["mag_err"] = flux_to_mag(df[flux_col].values, df[err_col].values)
    df = df.dropna(subset=["mag"])
    return df.sort_values("r:midpointMjdTai").reset_index(drop=True)


def lomb_scargle_period(
    t, mag, mag_err, pmin=PERIOD_MIN_DAYS, pmax=PERIOD_MAX_DAYS, samples_per_peak=LS_SAMPLES
):
    """Return (best_period, freq_grid, power, FAP_bootstrap)."""
    if len(t) < 5:
        return np.nan, None, None, np.nan
    ls = LombScargle(t, mag, mag_err)
    fmin = 1.0 / pmax
    fmax = 1.0 / pmin
    freq, power = ls.autopower(
        minimum_frequency=fmin,
        maximum_frequency=fmax,
        samples_per_peak=samples_per_peak,
    )
    best_freq = freq[np.argmax(power)]
    best_period = 1.0 / best_freq if best_freq > 0 else np.nan
    try:
        fap = ls.false_alarm_probability(power.max(), method="bootstrap", n_bootstraps=500)
    except Exception:
        fap = np.nan
    return best_period, freq, power, fap


def phase_fold(t, period, t0=0.0):
    """Return phase in [0, 1)."""
    return ((t - t0) % period) / period


print("Helper functions defined.")

In [ ]:
def mjd_to_datestr(mjd_array) -> list:
    """
    Convert an array of MJD (TAI) values to ISO date strings 'YYYY-MM-DD'.

    Uses astropy.time.Time for the conversion.
    """
    t = Time(np.asarray(mjd_array, dtype=float), format="mjd", scale="tai")
    return [tt.strftime("%Y-%m-%d") for tt in t]


def add_date_axis_on_top(ax, mjd_values: np.ndarray, n_ticks: int = 7) -> None:
    """
    Add a secondary x-axis on **top** of *ax* showing calendar dates (YYYY-MM-DD),
    inclined 40 degrees to the left for readability.
    """
    finite = mjd_values[np.isfinite(mjd_values)]
    if len(finite) < 2:
        return

    mjd_lo, mjd_hi = float(finite.min()), float(finite.max())
    if mjd_hi <= mjd_lo:
        return

    n_ticks = max(3, min(n_ticks, len(finite)))
    tick_mjd = np.linspace(mjd_lo, mjd_hi, n_ticks)
    tick_lbls = mjd_to_datestr(tick_mjd)

    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(tick_mjd)
    ax_top.set_xticklabels(tick_lbls, rotation=40, ha="left", fontsize=7)
    ax_top.tick_params(axis="x", length=4, pad=2)
    ax_top.set_xlabel("Date (UTC)", fontsize=12, labelpad=6)


print("mjd_to_datestr() and add_date_axis_on_top() defined.")

## 3. Reload summary DataFrames

In [ ]:
def _load_parquet(path: pathlib.Path, label: str) -> pd.DataFrame:
    if path.exists():
        df = pd.read_parquet(path)
        print(f"Loaded {label:25s}: {len(df):,} rows  |  cols: {list(df.columns)[:6]}...")
        return df
    print(f"NOT FOUND: {path}  ({label})")
    return pd.DataFrame()


df_obj = _load_parquet(DIR_DATA / "df_obj.parquet", "df_obj")
df_pulsators = _load_parquet(DIR_DATA / "df_pulsators.parquet", "df_pulsators")
df_periods = _load_parquet(DIR_DATA / "df_periods.parquet", "df_periods")
df_meta = _load_parquet(DIR_DATA / "lc_dict_meta.parquet", "lc_dict_meta")

print(f"\nTotal objects in survey    : {len(df_obj):,}")
print(f"Pulsating variable candidates: {len(df_pulsators):,}")
print(f"Objects with saved LC      : {len(df_meta):,}")

## 4. Reload individual light curves into `lc_dict`

In [ ]:
lc_dict = {}

if df_meta.empty:
    print("No metadata found — lc_dict will be empty.")
else:
    for _, row in df_meta.iterrows():
        oid = row["diaObjectId"]
        obj_dir = LC_DIR / str(oid)

        # Sources
        src_path = obj_dir / "sources.parquet"
        df_src = pd.read_parquet(src_path) if src_path.exists() else pd.DataFrame()

        # Forced photometry
        fp_path = obj_dir / "fp.parquet"
        df_fp = pd.read_parquet(fp_path) if fp_path.exists() else pd.DataFrame()

        # Metadata (all scalar columns except diaObjectId, n_src, n_fp)
        meta = row.drop(labels=["diaObjectId", "n_src", "n_fp"], errors="ignore").to_dict()

        lc_dict[oid] = {"src": df_src, "fp": df_fp, "meta": meta}

    print(f"Reloaded {len(lc_dict)} light curves from disk.")
    for oid, data in list(lc_dict.items())[:5]:
        print(
            f"  {oid}  src={len(data['src'])}  fp={len(data['fp'])}  "
            f"class={data['meta'].get('pulsator_class', '?')}"
        )

## 5. Summary statistics

In [ ]:
if not df_pulsators.empty:
    print("=" * 60)
    print("PULSATING VARIABLE CANDIDATES")
    print("=" * 60)
    if "pulsator_class" in df_pulsators.columns:
        print(df_pulsators["pulsator_class"].value_counts().to_string())
    if "field" in df_pulsators.columns:
        print("\nBy field:")
        print(df_pulsators["field"].value_counts().to_string())
    display(df_pulsators.head(10))
else:
    print("df_pulsators is empty.")

## 6. Raw light curves — overview plot

In [ ]:
if not lc_dict:
    print("No light curves available.")
else:
    NC_PLOT = min(50, len(lc_dict))
    ncols = 3
    nrows = int(np.ceil(NC_PLOT / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for idx, (oid, data) in enumerate(list(lc_dict.items())[:NC_PLOT]):
        ax = axes[idx]
        meta = data["meta"]
        df_filt = filter_lc(data["src"]) if not data["src"].empty else pd.DataFrame()

        if df_filt.empty:
            ax.text(0.5, 0.5, "No valid data", ha="center", va="center", transform=ax.transAxes)
        else:
            mag = df_filt["mag"].values
            mag = mag[np.isfinite(mag)]
            mag_min = np.percentile(mag, 5) - 0.3
            mag_max = np.percentile(mag, 95) + 0.3

            for band in BANDS:
                dfb = df_filt[df_filt["r:band"] == band]
                if dfb.empty:
                    continue
                ax.errorbar(
                    dfb["r:midpointMjdTai"],
                    dfb["mag"],
                    dfb["mag_err"],
                    fmt="o",
                    ms=4,
                    lw=0.5,
                    color=BAND_COLORS.get(band, "grey"),
                    label=band,
                    alpha=0.8,
                )
            # ax.invert_yaxis()
            ax.set_ylim(mag_max, mag_min)
            add_date_axis_on_top(ax, df_filt["r:midpointMjdTai"])

        pclass = meta.get("pulsator_class", "?")
        stype = meta.get("f:xm_simbad_otype", "?")
        vsx = meta.get("f:xm_vsx_Type", "?")
        ax.set_title(f"{oid}\n{pclass} | SIMBAD:{stype} VSX:{vsx}", fontsize=7)
        ax.set_xlabel("MJD", fontsize=7)
        ax.set_ylabel("AB mag", fontsize=7)
        ax.legend(fontsize=6, ncol=3)

    for ax in axes[NC_PLOT:]:
        ax.set_visible(False)

    plt.suptitle("Raw light curves — selected pulsators (reloaded)", y=1.01, fontsize=10)
    plt.tight_layout()
    savefig("pulsators_raw_lc")
    plt.show()

## 7. Forced-photometry light curves

In [ ]:
objects_with_fp = {oid: data for oid, data in lc_dict.items() if not data["fp"].empty}
print(f"{len(objects_with_fp)} objects have forced-photometry data.")

if objects_with_fp:
    NC_PLOT = min(100, len(objects_with_fp))
    ncols = 4
    nrows = int(np.ceil(NC_PLOT / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for idx, (oid, data) in enumerate(list(objects_with_fp.items())[:NC_PLOT]):
        ax = axes[idx]
        meta = data["meta"]

        df_src = filter_lc(data["src"]) if not data["src"].empty else pd.DataFrame()
        df_fp = data["fp"].copy()

        mag = df_filt["mag"].values
        mag = mag[np.isfinite(mag)]
        mag_min = np.percentile(mag, 5) - 0.3
        mag_max = np.percentile(mag, 95) + 0.3

        for band in BANDS:
            dfb = df_src[df_src["r:band"] == band]
            if dfb.empty:
                continue
            ax.errorbar(
                dfb["r:midpointMjdTai"],
                dfb["mag"],
                dfb["mag_err"],
                fmt="o",
                ms=4,
                lw=0.5,
                color=BAND_COLORS.get(band, "grey"),
                label=band,
                alpha=0.8,
            )
            # ax.invert_yaxis()
        ax.set_ylim(mag_max, mag_min)
        add_date_axis_on_top(ax, df_src["r:midpointMjdTai"])

        # Determine flux column available in FP data
        flux_col = None
        for col in ("r:psfFlux", "forcedSourceFlux", "fp_flux"):
            if col in df_fp.columns:
                flux_col = col
                break

        if flux_col is None or "r:midpointMjdTai" not in df_fp.columns:
            ax.text(
                0.5, 0.5, "No usable FP columns", ha="center", va="center", transform=ax.transAxes, fontsize=7
            )
        else:
            err_col = flux_col.replace("Flux", "FluxErr").replace("flux", "fluxErr")
            if err_col not in df_fp.columns:
                err_col = flux_col  # fallback
            band_col = "r:band" if "r:band" in df_fp.columns else None

            if band_col:
                for band in BANDS:
                    dfb = df_fp[df_fp[band_col] == band]
                    if dfb.empty:
                        continue
                    mag, mag_err = flux_to_mag(dfb[flux_col].values, dfb[err_col].values)
                    mask = np.isfinite(mag)
                    ax.errorbar(
                        dfb["r:midpointMjdTai"].values[mask],
                        mag[mask],
                        mag_err[mask],
                        fmt="s",
                        ms=2,
                        lw=0.5,
                        markerfacecolor="white",
                        markeredgecolor=BAND_COLORS.get(band, "grey"),
                        markeredgewidth=1.3,
                        # color=BAND_COLORS.get(band, "grey"),
                        label=band,
                        alpha=0.7,
                    )
            else:
                mag, mag_err = flux_to_mag(df_fp[flux_col].values, df_fp[err_col].values)
                mask = np.isfinite(mag)
                ax.errorbar(
                    df_fp["r:midpointMjdTai"].values[mask],
                    mag[mask],
                    mag_err[mask],
                    fmt="s",
                    ms=2,
                    # color="grey",
                    markerfacecolor="white",
                    markeredgecolor="grey",
                    markeredgewidth=1.3,
                    alpha=0.7,
                )
            # ax.invert_yaxis()

        pclass = meta.get("pulsator_class", "?")
        ax.set_title(f"{oid}\n{pclass}", fontsize=7)
        ax.set_xlabel("MJD", fontsize=7)
        ax.set_ylabel("AB mag", fontsize=7)
        ax.legend(fontsize=6, ncol=3)

    for ax in axes[NC_PLOT:]:
        ax.set_visible(False)

    plt.suptitle("Forced-photometry light curves (reloaded)", y=1.01, fontsize=10)
    plt.tight_layout()
    savefig("pulsators_fp_lc")
    plt.show()

## 8. Period search (recompute or reload)

If `df_periods.parquet` was saved by notebook 02, it is used directly.
Otherwise periods are recomputed from the reloaded sources.

In [ ]:
if df_periods.empty:
    print("df_periods not found — recomputing Lomb-Scargle periods ...")
    period_results = []

    for oid, data in lc_dict.items():
        if data["src"].empty:
            continue
        df_filt = filter_lc(data["src"])
        if len(df_filt) < 6:
            continue
        meta = data["meta"]

        # Prefer r-band; fall back to all bands
        df_r = df_filt[df_filt["r:band"] == "r"]
        if len(df_r) < 5:
            df_r = df_filt

        best_period, _, _, fap = lomb_scargle_period(
            df_r["r:midpointMjdTai"].values,
            df_r["mag"].values,
            df_r["mag_err"].values,
        )
        period_results.append(
            {
                "diaObjectId": oid,
                "field": meta.get("field", ""),
                "pulsator_class": meta.get("pulsator_class", ""),
                "simbad_otype": meta.get("f:xm_simbad_otype", ""),
                "vsx_type": meta.get("f:xm_vsx_Type", ""),
                "gcvs_type": meta.get("f:xm_gcvs_type", ""),
                "best_period_d": best_period,
                "ls_fap": fap,
                "n_pts": len(df_r),
            }
        )
        print(f"  {oid}  P={best_period:.3f}d  FAP={fap:.1e}  ({meta.get('pulsator_class', '?')})")

    df_periods = pd.DataFrame(period_results)
    if not df_periods.empty:
        df_periods.to_parquet(DIR_DATA / "df_periods.parquet", index=False)
        print(f"Saved recomputed df_periods ({len(df_periods)} rows).")
else:
    print(f"Using precomputed df_periods ({len(df_periods)} rows).")

if not df_periods.empty:
    display(df_periods.sort_values("best_period_d"))

## 9. Phase-folded light curves

In [ ]:
if df_periods.empty:
    print("No period data available.")
else:
    df_plot = df_periods.dropna(subset=["best_period_d"]).sort_values("ls_fap").head(12)
    ncols = 3
    nrows = int(np.ceil(len(df_plot) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for idx, (_, row) in enumerate(df_plot.iterrows()):
        oid = row["diaObjectId"]
        period = row["best_period_d"]
        data = lc_dict.get(oid, {})
        if not data or data["src"].empty:
            continue
        df_filt = filter_lc(data["src"])
        if df_filt.empty:
            continue

        ax = axes[idx]
        t0 = df_filt["r:midpointMjdTai"].min()
        for band in BANDS:
            dfb = df_filt[df_filt["r:band"] == band]
            if dfb.empty:
                continue
            phi = phase_fold(dfb["r:midpointMjdTai"].values, period, t0)
            ax.errorbar(
                np.concatenate([phi, phi + 1]),
                np.concatenate([dfb["mag"].values] * 2),
                yerr=np.concatenate([dfb["mag_err"].values] * 2),
                fmt="o",
                ms=3,
                lw=0.5,
                color=BAND_COLORS.get(band, "grey"),
                label=band,
                alpha=0.85,
            )
        ax.set_xlim(0, 2)
        ax.invert_yaxis()
        ax.set_xlabel("Phase")
        ax.set_ylabel("AB mag")
        ax.set_title(
            f"{oid}  P={period:.3f}d  FAP={row['ls_fap']:.1e}\n"
            f"{row['pulsator_class']} | {row.get('vsx_type', '?')}",
            fontsize=7,
        )
        ax.legend(fontsize=6, ncol=3)

    for ax in axes[len(df_plot) :]:
        ax.set_visible(False)

    plt.suptitle("Phase-folded light curves (sorted by LS FAP) — reloaded", y=1.01, fontsize=10)
    plt.tight_layout()
    savefig("pulsators_phased_lc")
    plt.show()

## 10. Lomb-Scargle power spectrum for individual objects

In [ ]:
# Show LS periodogram for the top-N most significant objects
if not df_periods.empty:
    TOP_N = 6
    df_top = df_periods.dropna(subset=["best_period_d"]).sort_values("ls_fap").head(TOP_N)
    ncols = 2
    nrows = int(np.ceil(TOP_N / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 3.5 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for idx, (_, row) in enumerate(df_top.iterrows()):
        oid = row["diaObjectId"]
        data = lc_dict.get(oid, {})
        if not data or data["src"].empty:
            continue
        df_filt = filter_lc(data["src"])
        if len(df_filt) < 5:
            continue

        df_r = df_filt[df_filt["r:band"] == "r"]
        if len(df_r) < 5:
            df_r = df_filt

        _, freq, power, fap = lomb_scargle_period(
            df_r["r:midpointMjdTai"].values,
            df_r["mag"].values,
            df_r["mag_err"].values,
        )
        if freq is None:
            continue

        ax = axes[idx]
        period_grid = 1.0 / freq
        ax.semilogx(period_grid, power, lw=0.8, color="steelblue")
        ax.axvline(row["best_period_d"], color="red", lw=1.5, ls="--", label=f"P={row['best_period_d']:.3f}d")
        ax.set_xlabel("Period (days)")
        ax.set_ylabel("LS power")
        ax.set_title(f"{oid}  FAP={row['ls_fap']:.1e}\n{row['pulsator_class']}", fontsize=8)
        ax.legend(fontsize=7)

    for ax in axes[TOP_N:]:
        ax.set_visible(False)

    plt.suptitle("Lomb-Scargle periodograms — top objects", y=1.01, fontsize=10)
    plt.tight_layout()
    savefig("pulsators_ls_periodogram")
    plt.show()
else:
    print("No period data available.")

## 11. Period–Luminosity diagram

In [ ]:
if df_periods.empty or df_periods["best_period_d"].isna().all():
    print("No period data available.")
else:
    # Add median r-band magnitude from reloaded sources
    mag_med = []
    for oid in df_periods["diaObjectId"]:
        data = lc_dict.get(oid, {})
        if data and not data["src"].empty:
            df_r = filter_lc(data["src"])
            df_r = df_r[df_r["r:band"] == "r"]
            mag_med.append(np.nanmedian(df_r["mag"]) if len(df_r) > 0 else np.nan)
        else:
            mag_med.append(np.nan)

    df_periods = df_periods.copy()
    df_periods["mag_r_median"] = mag_med

    df_pl = df_periods.dropna(subset=["best_period_d", "mag_r_median"])
    df_pl = df_pl[(df_pl["best_period_d"] > 0.1) & (df_pl["best_period_d"] < 200)]

    class_marker = {
        "cepheid_classical": ("*", "red", 100),
        "cepheid_type2": ("^", "darkorange", 80),
        "rr_lyrae": ("o", "dodgerblue", 40),
        "delta_scuti": ("s", "green", 30),
        "lpv_mira": ("D", "purple", 40),
        "rv_tauri": ("P", "brown", 50),
        "other_pulsator": ("x", "grey", 25),
    }

    fig, ax = plt.subplots(figsize=(8, 6))
    for cls, (marker, color, size) in class_marker.items():
        sub = df_pl[df_pl["pulsator_class"] == cls]
        if sub.empty:
            continue
        ax.scatter(
            np.log10(sub["best_period_d"]),
            sub["mag_r_median"],
            marker=marker,
            color=color,
            s=size,
            alpha=0.8,
            label=f"{cls} (N={len(sub)})",
            zorder=3,
        )

    # Leavitt law reference (rough)
    logP = np.linspace(0, 2, 50)
    ax.plot(logP, -2.81 * logP + 11.5, "k--", lw=1, alpha=0.4, label="Leavitt law (rough, DM≈11)")

    ax.set_xlabel("log₁₀(Period / days)")
    ax.set_ylabel("Median r-band AB mag (apparent)")
    ax.invert_yaxis()
    ax.set_title(
        "Period–Luminosity diagram — Pulsating variables in LSST fields\n"
        "(apparent magnitudes, no distance or extinction correction)"
    )
    ax.legend(fontsize=8, loc="best")
    ax.set_ylim(26, 16)
    plt.tight_layout()
    savefig("pulsators_PL_diagram")
    plt.show()

## 12. Interactive single-object explorer

Three-panel view for the best-detected object (lowest LS FAP).
**Marker convention** — same band colour throughout:

| Symbol | Meaning |
|--------|---------|
| filled circle `●` | DIA detection (sources) |
| open circle `○` (white face, coloured edge) | Forced photometry |

- **Topt** : light curve (src + FP on the same axes, vs MJD)
- **Bottom-left** : Lomb-Scargle periodogram
- **Bottom-right** : phase-folded light curve (src + FP on the same axes, × 2 periods)

Set `EXPLORE_OID` to any `diaObjectId` present in `lc_dict` to inspect a different object.


In [ ]:
# ── Section 12: single-object explorer ───────────────────────────────────────


def _fp_mag_by_band(df_fp):
    """
    Extract per-band (mjd, mag, mag_err) arrays from a forced-photometry DataFrame.
    Returns dict {band: (mjd_arr, mag_arr, mag_err_arr)}, empty if data absent.
    """
    result = {}
    if df_fp.empty or "r:midpointMjdTai" not in df_fp.columns:
        return result
    flux_col = next(
        (c for c in ("r:psfFlux", "forcedSourceFlux", "fp_flux") if c in df_fp.columns),
        None,
    )
    if flux_col is None:
        return result
    err_col = flux_col.replace("Flux", "FluxErr").replace("flux", "fluxErr")
    if err_col not in df_fp.columns:
        err_col = flux_col
    band_col = "r:band" if "r:band" in df_fp.columns else None
    rows = [df_fp] if band_col is None else [(band, df_fp[df_fp[band_col] == band]) for band in BANDS]
    for item in rows:
        if band_col is None:
            band, dfb = "?", item
        else:
            band, dfb = item
        if dfb.empty:
            continue
        mag, mag_err = flux_to_mag(dfb[flux_col].values, dfb[err_col].values)
        mask = np.isfinite(mag)
        if mask.any():
            result[band] = (
                dfb["r:midpointMjdTai"].values[mask],
                mag[mask],
                mag_err[mask],
            )
    return result

In [ ]:
if not lc_dict:
    print("lc_dict is empty — nothing to explore.")
else:
    # ── Object selection ──────────────────────────────────────────────────────
    # Default: best LS detection.  Override: EXPLORE_OID = <some diaObjectId>
    if not df_periods.empty and "ls_fap" in df_periods.columns:
        EXPLORE_OID = df_periods.dropna(subset=["ls_fap"]).sort_values("ls_fap")["diaObjectId"].iloc[0]
    else:
        EXPLORE_OID = next(iter(lc_dict))

    print(f"Exploring object: {EXPLORE_OID}")
    data = lc_dict[EXPLORE_OID]
    meta = data["meta"]
    df_src = filter_lc(data["src"]) if not data["src"].empty else pd.DataFrame()
    fp_by_band = _fp_mag_by_band(data["fp"])

    n_src = len(df_src)
    n_fp = sum(len(v[0]) for v in fp_by_band.values())
    print(f"  src detections : {n_src}")
    print(f"  FP epochs      : {n_fp}")

    # ── Figure layout: 2×2 grid, bottom row merged ────────────────────────────
    fig = plt.figure(figsize=(12, 8))
    gs = gridspec.GridSpec(2, 2)
    # Top row (merged)
    ax_lc = fig.add_subplot(gs[0, :])  # top  : light curve
    # Bottom row
    ax_ls = fig.add_subplot(gs[1, 0])  # bottom-left : LS periodogram
    ax_ph = fig.add_subplot(gs[1, 1])  # bottom-right : phase (full width)

    mag_min = 16
    mag_max = 27
    # ── Panel 1 — Light curve: src (filled ●) + FP (open ○) ─────────────────
    if not df_src.empty:
        mag = df_src["mag"].values
        mag = mag[np.isfinite(mag)]
        mag_min = np.percentile(mag, 5) - 1.0
        mag_max = np.percentile(mag, 95) + 2.0

        for band in BANDS:
            dfb = df_src[df_src["r:band"] == band]
            if dfb.empty:
                continue
            col = BAND_COLORS.get(band, "grey")
            ax_lc.errorbar(
                dfb["r:midpointMjdTai"],
                dfb["mag"],
                dfb["mag_err"],
                fmt="o",
                ms=4,
                lw=0.5,
                color=col,
                markerfacecolor=col,
                markeredgecolor=col,
                alpha=0.85,
                label=f"{band} src",
            )

    for band, (mjd, mag, mag_err) in fp_by_band.items():
        col = BAND_COLORS.get(band, "grey")
        ax_lc.errorbar(
            mjd,
            mag,
            mag_err,
            fmt="o",
            ms=5,
            lw=0.5,
            color=col,
            markerfacecolor="white",
            markeredgecolor=col,
            markeredgewidth=1.3,
            alpha=0.80,
            label=f"{band} FP",
        )

    if not df_src.empty or fp_by_band:
        pass
        # ax_lc.invert_yaxis()

    ax_lc.set_xlabel("MJD")
    ax_lc.set_ylabel("AB mag")
    ax_lc.set_title(f"Light curve — {EXPLORE_OID}")
    ax_lc.legend(
        fontsize=10,
        ncol=6,
        title="● src   ○ forced phot.",
        title_fontsize=10,
        loc="best",
        framealpha=0.7,
    )
    ax_lc.set_ylim(mag_max, mag_min)
    add_date_axis_on_top(ax_lc, df_src["r:midpointMjdTai"])

    # ── Panel 2 — Lomb-Scargle periodogram ───────────────────────────────────
    if not df_src.empty and len(df_src) >= 5:
        df_r = df_src[df_src["r:band"] == "r"]
        if len(df_r) < 5:
            df_r = df_src
        best_period, freq, power, fap = lomb_scargle_period(
            df_r["r:midpointMjdTai"].values,
            df_r["mag"].values,
            df_r["mag_err"].values,
        )
        if freq is not None:
            ax_ls.semilogx(1.0 / freq, power, lw=0.8, color="steelblue")
            if np.isfinite(best_period):
                ax_ls.axvline(
                    best_period,
                    color="red",
                    lw=1.5,
                    ls="--",
                    label=f"P = {best_period:.4f} d\nFAP = {fap:.2e}",
                )
            ax_ls.legend(fontsize=8)
    ax_ls.set_xlabel("Period (days)")
    ax_ls.set_ylabel("LS power")
    ax_ls.set_title("Lomb-Scargle periodogram (r-band or all)")

    # ── Panel 3 — Phase-folded: src (filled ●) + FP (open ○) ────────────────
    row_p = df_periods[df_periods["diaObjectId"] == EXPLORE_OID] if not df_periods.empty else pd.DataFrame()
    if not row_p.empty and np.isfinite(row_p.iloc[0]["best_period_d"]):
        P = row_p.iloc[0]["best_period_d"]
        t0 = df_src["r:midpointMjdTai"].min() if not df_src.empty else 0.0

        # Sources — filled circles
        if not df_src.empty:
            for band in BANDS:
                dfb = df_src[df_src["r:band"] == band]
                if dfb.empty:
                    continue
                col = BAND_COLORS.get(band, "grey")
                phi = phase_fold(dfb["r:midpointMjdTai"].values, P, t0)
                phi2 = np.concatenate([phi, phi + 1])
                mag2 = np.concatenate([dfb["mag"].values] * 2)
                err2 = np.concatenate([dfb["mag_err"].values] * 2)
                ax_ph.errorbar(
                    phi2,
                    mag2,
                    err2,
                    fmt="o",
                    ms=4,
                    lw=0.5,
                    color=col,
                    markerfacecolor=col,
                    markeredgecolor=col,
                    alpha=0.85,
                    label=f"{band} src",
                )

        # Forced photometry — open circles
        for band, (mjd, mag, mag_err) in fp_by_band.items():
            col = BAND_COLORS.get(band, "grey")
            phi = phase_fold(mjd, P, t0)
            phi2 = np.concatenate([phi, phi + 1])
            mag2 = np.concatenate([mag, mag])
            err2 = np.concatenate([mag_err, mag_err])
            ax_ph.errorbar(
                phi2,
                mag2,
                err2,
                fmt="o",
                ms=5,
                lw=0.5,
                color=col,
                markerfacecolor="white",
                markeredgecolor=col,
                markeredgewidth=1.3,
                alpha=0.80,
                label=f"{band} FP",
            )

        ax_ph.set_xlim(0, 2)
        ax_ph.invert_yaxis()
        ax_ph.set_title(
            f"Phase-folded light curve  —  P = {P:.4f} d  (× 2 periods)",
            fontsize=11,
        )
        ax_ph.legend(
            fontsize=7,
            ncol=6,
            title="● src   ○ forced phot.",
            title_fontsize=7,
            loc="best",
            framealpha=0.7,
        )
    else:
        ax_ph.text(
            0.5,
            0.5,
            "No period available",
            ha="center",
            va="center",
            transform=ax_ph.transAxes,
            fontsize=12,
        )
    ax_ph.set_xlabel("Phase", fontsize=11)
    ax_ph.set_ylabel("AB mag", fontsize=11)
    ax_ph.set_ylim(mag_max, mag_min)

    # ── Suptitle ──────────────────────────────────────────────────────────────
    pclass = meta.get("pulsator_class", "?")
    stype = meta.get("f:xm_simbad_otype", "?")
    vsx = meta.get("f:xm_vsx_Type", "?")
    plt.suptitle(
        f"{EXPLORE_OID}  —  {pclass}  |  SIMBAD: {stype}  |  VSX: {vsx}\n"
        f"src detections: {n_src}   FP epochs: {n_fp}",
        fontsize=11,
        y=1.01,
    )
    plt.tight_layout()
    savefig(f"explorer_{EXPLORE_OID}")
    plt.show()

## 13. Period histograms by pulsator class

Do the different variable-star families separate cleanly in period space?  
This matters for photometric-calibration work: we want families with
**well-defined, narrow period ranges** so that a phase-folded flux model
can be constructed and compared to new observations.

Known period ranges (literature):

| Class | Typical period range |
|---|---|
| Delta Scuti | 0.02 – 0.3 d |
| RR Lyrae (ab) | 0.3 – 1.0 d |
| RR Lyrae (c)  | 0.2 – 0.5 d |
| W Vir / Type II Cep | 1 – 35 d |
| Classical Cepheids (DCEP) | 1 – 100 d |
| Mira / LPV | 100 – 1000 d |


In [ ]:
# ── Section 13: period histograms ────────────────────────────────────────────
import matplotlib.patches as mpatches

# Reference period ranges (days) for vertical shading
PERIOD_BANDS_REF = [
    (0.02, 0.30, "#f0e68c", "Delta Scuti"),
    (0.30, 1.00, "#add8e6", "RR Lyrae"),
    (1.00, 35.0, "#90ee90", "W Vir / Type II Cep"),
    (1.00, 100.0, "#ffa07a", "Classical Cepheids"),
    (100.0, 500.0, "#d8bfd8", "Mira / LPV"),
]

CLASS_COLORS = {
    "cepheid_classical": "red",
    "cepheid_type2": "darkorange",
    "rr_lyrae": "dodgerblue",
    "delta_scuti": "green",
    "lpv_mira": "purple",
    "rv_tauri": "brown",
    "other_pulsator": "grey",
}

if df_periods.empty or "best_period_d" not in df_periods.columns:
    print("No period data — skipping histograms.")
else:
    df_p = df_periods.dropna(subset=["best_period_d"]).copy()
    df_p = df_p[df_p["best_period_d"] > 0]
    classes_present = [c for c in CLASS_COLORS if c in df_p["pulsator_class"].values]

    # ── Figure 1: stacked histogram on log-period axis ────────────────────────
    fig, axes = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={"height_ratios": [3, 1.5]})

    ax_hist = axes[0]
    ax_strip = axes[1]

    bins = np.logspace(
        np.log10(max(0.01, df_p["best_period_d"].min() * 0.8)),
        np.log10(min(600.0, df_p["best_period_d"].max() * 1.2)),
        40,
    )

    # Reference background bands
    for plo, phi, col, label in PERIOD_BANDS_REF:
        ax_hist.axvspan(plo, phi, alpha=0.10, color=col, zorder=0)
        ax_strip.axvspan(plo, phi, alpha=0.10, color=col, zorder=0)

    # Stacked histogram
    data_stacked = [df_p[df_p["pulsator_class"] == c]["best_period_d"].values for c in classes_present]
    colors_stacked = [CLASS_COLORS[c] for c in classes_present]

    ax_hist.hist(
        data_stacked,
        bins=bins,
        stacked=True,
        color=colors_stacked,
        label=classes_present,
        alpha=0.85,
        edgecolor="white",
        linewidth=0.4,
    )
    ax_hist.set_xscale("log")
    ax_hist.set_xlabel("Period (days)", fontsize=11)
    ax_hist.set_ylabel("Number of objects", fontsize=11)
    ax_hist.set_title("Period distribution by pulsator class", fontsize=12)
    ax_hist.legend(fontsize=9, loc="upper right")
    ax_hist.set_yscale("log")

    # Secondary x-axis: log10 P
    ax2 = ax_hist.twiny()
    ax2.set_xscale("log")
    ax2.set_xlim(ax_hist.get_xlim())
    ax2.set_xlabel("log₁₀(Period / days)", fontsize=9)
    logP_ticks = [0.03, 0.1, 0.3, 1, 3, 10, 30, 100, 300]
    ax2.set_xticks([t for t in logP_ticks if ax_hist.get_xlim()[0] <= t <= ax_hist.get_xlim()[1]])
    ax2.set_xticklabels(
        [f"{np.log10(t):.1f}" for t in logP_ticks if ax_hist.get_xlim()[0] <= t <= ax_hist.get_xlim()[1]],
        fontsize=8,
    )

    # ── Strip plot: individual periods per class (jittered) ──────────────────
    for k, cls in enumerate(classes_present):
        vals = df_p[df_p["pulsator_class"] == cls]["best_period_d"].values
        if len(vals) == 0:
            continue
        jitter = np.random.default_rng(42).uniform(-0.3, 0.3, len(vals))
        ax_strip.scatter(
            vals,
            np.full(len(vals), k) + jitter,
            color=CLASS_COLORS[cls],
            s=20,
            alpha=0.7,
            zorder=3,
        )
    ax_strip.set_xscale("log")
    ax_strip.set_yticks(range(len(classes_present)))
    ax_strip.set_yticklabels(classes_present, fontsize=9)
    ax_strip.set_xlabel("Period (days)", fontsize=11)
    ax_strip.set_title("Individual periods (strip plot)", fontsize=10)
    ax_strip.set_xlim(ax_hist.get_xlim())

    plt.tight_layout()
    savefig("period_histogram_by_class")
    plt.show()

    # ── Figure 2: per-class KDE on log P ─────────────────────────────────────
    try:
        from scipy.stats import gaussian_kde

        HAS_SCIPY = True
    except ImportError:
        HAS_SCIPY = False
        print("scipy not available — KDE plot skipped")

    if HAS_SCIPY:
        log_bins = np.linspace(
            np.log10(max(0.01, df_p["best_period_d"].min() * 0.8)),
            np.log10(min(600.0, df_p["best_period_d"].max() * 1.2)),
            300,
        )
        fig, ax = plt.subplots(figsize=(10, 4))
        for plo, phi, col, label in PERIOD_BANDS_REF:
            ax.axvspan(np.log10(plo), np.log10(phi), alpha=0.09, color=col, zorder=0)
        for cls in classes_present:
            vals = np.log10(df_p[df_p["pulsator_class"] == cls]["best_period_d"].values)
            if len(vals) < 3:
                continue
            kde = gaussian_kde(vals, bw_method="scott")
            ax.plot(log_bins, kde(log_bins), color=CLASS_COLORS[cls], lw=2, label=f"{cls} (N={len(vals)})")
            ax.fill_between(log_bins, kde(log_bins), alpha=0.15, color=CLASS_COLORS[cls])
        ax.set_xlabel("log₁₀(Period / days)", fontsize=11)
        ax.set_ylabel("KDE density", fontsize=11)
        ax.set_title("KDE of period distribution by pulsator class", fontsize=12)
        ax.legend(fontsize=9)

        # Annotate reference bands
        for plo, phi, _, label in PERIOD_BANDS_REF:
            xmid = (np.log10(plo) + np.log10(phi)) / 2
            xlim = ax.get_xlim()
            if xlim[0] < xmid < xlim[1]:
                ax.text(
                    xmid,
                    ax.get_ylim()[1] * 0.95,
                    label,
                    ha="center",
                    va="top",
                    fontsize=7,
                    color="#555",
                    style="italic",
                )

        plt.tight_layout()
        savefig("period_kde_by_class")
        plt.show()

    # ── Summary table ─────────────────────────────────────────────────────────
    print("\nPeriod statistics by pulsator class:")
    print("=" * 72)
    summary_rows = []
    for cls in CLASS_COLORS:
        sub = df_p[df_p["pulsator_class"] == cls]["best_period_d"]
        if len(sub) == 0:
            continue
        summary_rows.append(
            {
                "class": cls,
                "N": len(sub),
                "P_min_d": sub.min(),
                "P_median_d": sub.median(),
                "P_max_d": sub.max(),
                "P_std_d": sub.std(),
            }
        )
    df_summary = pd.DataFrame(summary_rows)
    display(df_summary.round(3))

## 10A. Callable plot functions

Two reusable functions driven by `oid`:

* **`plot_lc_single(oid)`** — one subplot: src (filled ●) + FP (open ○),
  y-axis clipped to `[mag_max, mag_min]`, twinx date axis on top.
* **`plot_lc_full(oid)`** — three subplots: light curve (src+FP),
  Lomb-Scargle periodogram, phase-folded diagram.


In [ ]:
def _prepare_fp_by_band(df_fp):
    """
    Parse forced-photometry DataFrame into a dict
    {band: (mjd_array, mag_array, mag_err_array)}.
    Returns an empty dict if df_fp has no usable columns.
    """
    fp_by_band = {}
    if df_fp.empty:
        return fp_by_band

    # Detect flux column
    flux_col = None
    for col in ("r:psfFlux", "forcedSourceFlux", "fp_flux"):
        if col in df_fp.columns:
            flux_col = col
            break
    if flux_col is None or "r:midpointMjdTai" not in df_fp.columns:
        return fp_by_band

    err_col = flux_col.replace("Flux", "FluxErr").replace("flux", "fluxErr")
    if err_col not in df_fp.columns:
        err_col = flux_col

    band_col = "r:band" if "r:band" in df_fp.columns else None
    bands_iter = BANDS if band_col else [None]

    for band in bands_iter:
        dfb = df_fp[df_fp[band_col] == band] if band_col else df_fp
        if dfb.empty:
            continue
        mag, mag_err = flux_to_mag(dfb[flux_col].values, dfb[err_col].values)
        mask = np.isfinite(mag)
        if mask.sum() == 0:
            continue
        fp_by_band[band or "?"] = (
            dfb["r:midpointMjdTai"].values[mask],
            mag[mask],
            mag_err[mask],
        )
    return fp_by_band


def _mag_limits(df_src, fp_by_band, margin=0.3):
    """
    Compute robust [mag_min, mag_max] range from src + FP data.
    Returns (mag_min, mag_max) so that ylim = (mag_max, mag_min) inverts the axis.
    """
    all_mags = []
    if not df_src.empty and "mag" in df_src.columns:
        all_mags.append(df_src["mag"].values)
    for _, (_, mg, _) in fp_by_band.items():
        all_mags.append(mg)
    if not all_mags:
        return 14.0, 22.0
    arr = np.concatenate(all_mags)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return 14.0, 22.0
    return float(np.percentile(arr, 5) - margin), float(np.percentile(arr, 95) + margin)


# ─────────────────────────────────────────────────────────────────────────────
# Function 1 : single-panel light curve (src + FP)
# ─────────────────────────────────────────────────────────────────────────────


def plot_lc_single(
    oid,
    ax=None,
    save=False,
    show=True,
):
    """
    Plot the light curve of *oid* (sources + forced photometry) in one subplot.

    Parameters
    ----------
    oid  : diaObjectId key present in lc_dict
    ax   : existing matplotlib Axes to draw into (None → create a new figure)
    save : if True, call savefig() with a per-object name
    show : if True, call plt.show()

    Returns
    -------
    ax : the Axes object
    """
    if oid not in lc_dict:
        print(f"[plot_lc_single] oid {oid} not in lc_dict — skipping.")
        return None

    data = lc_dict[oid]
    meta = data["meta"]
    df_src = filter_lc(data["src"]) if not data["src"].empty else pd.DataFrame()
    fp_by_band = _prepare_fp_by_band(data["fp"])
    mag_min, mag_max = _mag_limits(df_src, fp_by_band)

    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(11, 4.5))

    # ── Sources : filled circles ─────────────────────────────────────────────
    if not df_src.empty:
        for band in BANDS:
            dfb = df_src[df_src["r:band"] == band]
            if dfb.empty:
                continue
            col = BAND_COLORS.get(band, "grey")
            ax.errorbar(
                dfb["r:midpointMjdTai"],
                dfb["mag"],
                dfb["mag_err"],
                fmt="o",
                ms=5,
                lw=0.5,
                color=col,
                markerfacecolor=col,
                markeredgecolor=col,
                alpha=0.85,
                label=f"{band} src",
            )

    # ── Forced photometry : open circles ────────────────────────────────────
    for band, (mjd, mag, mag_err) in fp_by_band.items():
        col = BAND_COLORS.get(band, "grey")
        ax.errorbar(
            mjd,
            mag,
            mag_err,
            fmt="o",
            ms=5,
            lw=0.5,
            color=col,
            markerfacecolor="white",
            markeredgecolor=col,
            markeredgewidth=1.3,
            alpha=0.80,
            label=f"{band} FP",
        )

    ax.set_ylim(mag_max, mag_min)  # inverted axis
    ax.set_xlabel("MJD", fontsize=11)
    ax.set_ylabel("AB mag", fontsize=11)
    pclass = meta.get("pulsator_class", "?")
    stype = meta.get("f:xm_simbad_otype", "?")
    vsx = meta.get("f:xm_vsx_Type", "?")
    ax.set_title(
        f"{oid}  |  {pclass}  |  SIMBAD: {stype}  |  VSX: {vsx}",
        fontsize=10,
    )
    ax.legend(
        fontsize=8, ncol=6, title="● src   ○ forced phot.", title_fontsize=8, loc="best", framealpha=0.7
    )

    # ── Date axis on top (twinx) ─────────────────────────────────────────────
    all_mjd = []
    if not df_src.empty:
        all_mjd.append(df_src["r:midpointMjdTai"].values)
    for _, (mjd, _, _) in fp_by_band.items():
        all_mjd.append(mjd)
    if all_mjd:
        add_date_axis_on_top(ax, np.concatenate(all_mjd))

    if own_fig:
        plt.tight_layout()
        if save:
            savefig(f"lc_single_{oid}")
        if show:
            plt.show()
    return ax


# ─────────────────────────────────────────────────────────────────────────────
# Function 2 : full 3-panel figure (LC + Lomb-Scargle + phase)
# ─────────────────────────────────────────────────────────────────────────────


def plot_lc_full(
    oid,
    period_override=None,
    save=False,
    show=True,
):
    """
    Three-panel figure for *oid*:
      Panel 1 — light curve (src + FP)
      Panel 2 — Lomb-Scargle periodogram
      Panel 3 — phase-folded light curve (src + FP)

    Parameters
    ----------
    oid             : diaObjectId key present in lc_dict
    period_override : float or None — if given, use this period instead of
                      the value in df_periods (useful for testing)
    save            : save figure to DIR_FIGS
    show            : call plt.show()

    Returns
    -------
    (fig, axes) : tuple
    """
    if oid not in lc_dict:
        print(f"[plot_lc_full] oid {oid} not in lc_dict — skipping.")
        return None, None

    data = lc_dict[oid]
    meta = data["meta"]
    df_src = filter_lc(data["src"]) if not data["src"].empty else pd.DataFrame()
    fp_by_band = _prepare_fp_by_band(data["fp"])
    mag_min, mag_max = _mag_limits(df_src, fp_by_band)

    n_src = len(data["src"])
    n_fp = len(data["fp"])

    fig = plt.figure(figsize=(12, 8))
    gs = gridspec.GridSpec(2, 2, hspace=0.45)
    ax_lc = fig.add_subplot(gs[0, :])
    ax_ls = fig.add_subplot(gs[1, 0])
    ax_ph = fig.add_subplot(gs[1, 1])

    # ── Panel 1 : light curve ────────────────────────────────────────────────
    plot_lc_single(oid, ax=ax_lc, save=False, show=False)

    # ── Panel 2 : Lomb-Scargle periodogram ───────────────────────────────────
    best_period_ls = np.nan
    freq = power = None
    fap = np.nan
    if not df_src.empty and len(df_src) >= 5:
        df_r = df_src[df_src["r:band"] == "r"]
        if len(df_r) < 5:
            df_r = df_src
        best_period_ls, freq, power, fap = lomb_scargle_period(
            df_r["r:midpointMjdTai"].values,
            df_r["mag"].values,
            df_r["mag_err"].values,
        )
    if freq is not None:
        ax_ls.semilogx(1.0 / freq, power, lw=0.8, color="steelblue")
        if np.isfinite(best_period_ls):
            ax_ls.axvline(
                best_period_ls,
                color="red",
                lw=1.5,
                ls="--",
                label=f"P = {best_period_ls:.4f} d   FAP = {fap:.2e}",
            )
        ax_ls.legend(fontsize=9)
    else:
        ax_ls.text(0.5, 0.5, "Not enough data for LS", ha="center", va="center", transform=ax_ls.transAxes)
    ax_ls.set_xlabel("Period (days)", fontsize=11)
    ax_ls.set_ylabel("LS power", fontsize=11)
    ax_ls.set_title("Lomb-Scargle periodogram (r-band or all)", fontsize=11)

    # ── Determine best period for phase-folding ───────────────────────────────
    if period_override is not None:
        P = float(period_override)
    elif not df_periods.empty:
        row_p = df_periods[df_periods["diaObjectId"] == oid]
        P = row_p.iloc[0]["best_period_d"] if not row_p.empty else best_period_ls
    else:
        P = best_period_ls

    # ── Panel 3 : phase-folded ────────────────────────────────────────────────
    if np.isfinite(P) if P is not None else False:
        t0 = df_src["r:midpointMjdTai"].min() if not df_src.empty else 0.0

        # Sources — filled circles
        if not df_src.empty:
            for band in BANDS:
                dfb = df_src[df_src["r:band"] == band]
                if dfb.empty:
                    continue
                col = BAND_COLORS.get(band, "grey")
                phi = phase_fold(dfb["r:midpointMjdTai"].values, P, t0)
                phi2 = np.concatenate([phi, phi + 1])
                mag2 = np.concatenate([dfb["mag"].values] * 2)
                err2 = np.concatenate([dfb["mag_err"].values] * 2)
                ax_ph.errorbar(
                    phi2,
                    mag2,
                    err2,
                    fmt="o",
                    ms=4,
                    lw=0.5,
                    color=col,
                    markerfacecolor=col,
                    markeredgecolor=col,
                    alpha=0.85,
                    label=f"{band} src",
                )

        # Forced photometry — open circles
        for band, (mjd, mag, mag_err) in fp_by_band.items():
            col = BAND_COLORS.get(band, "grey")
            phi = phase_fold(mjd, P, t0)
            phi2 = np.concatenate([phi, phi + 1])
            ax_ph.errorbar(
                phi2,
                np.concatenate([mag, mag]),
                np.concatenate([mag_err, mag_err]),
                fmt="o",
                ms=5,
                lw=0.5,
                color=col,
                markerfacecolor="white",
                markeredgecolor=col,
                markeredgewidth=1.3,
                alpha=0.80,
                label=f"{band} FP",
            )

        ax_ph.set_xlim(0, 2)
        ax_ph.set_ylim(mag_max, mag_min)
        ax_ph.set_title(f"Phase-folded  —  P = {P:.4f} d  (× 2 periods)", fontsize=11)
        ax_ph.legend(
            fontsize=7, ncol=6, title="● src   ○ forced phot.", title_fontsize=7, loc="best", framealpha=0.7
        )
    else:
        ax_ph.text(
            0.5,
            0.5,
            "No valid period available",
            ha="center",
            va="center",
            transform=ax_ph.transAxes,
            fontsize=12,
        )
    ax_ph.set_xlabel("Phase", fontsize=11)
    ax_ph.set_ylabel("AB mag", fontsize=11)

    # ── Suptitle ─────────────────────────────────────────────────────────────
    pclass = meta.get("pulsator_class", "?")
    stype = meta.get("f:xm_simbad_otype", "?")
    vsx = meta.get("f:xm_vsx_Type", "?")
    fig.suptitle(
        f"{oid}  —  {pclass}  |  SIMBAD: {stype}  |  VSX: {vsx}\nsrc detections: {n_src}   FP epochs: {n_fp}",
        fontsize=11,
        y=1.01,
    )

    if save:
        savefig(f"lc_full_{oid}")
    if show:
        plt.show()
    return fig, (ax_lc, ax_ls, ax_ph)


print("plot_lc_single() and plot_lc_full() defined.")

## 10B. Selection helpers

Two helpers to build the list of `oid` values to loop over:

* **`select_by_class(classes)`** — filter `lc_dict` keys by `pulsator_class`
  (or SIMBAD / VSX type).
* **`select_by_period_range(p_min, p_max)`** — filter by period window
  (uses `df_periods`).


In [ ]:
def select_by_class(classes, use_simbad=False, use_vsx=False):
    """
    Return list of oid whose pulsator_class (or SIMBAD/VSX type) is in *classes*.

    Parameters
    ----------
    classes      : str or list of str — class names to keep
                   Examples: 'rr_lyrae', ['rr_lyrae', 'delta_scuti']
    use_simbad   : also match on f:xm_simbad_otype
    use_vsx      : also match on f:xm_vsx_Type

    Returns
    -------
    list of diaObjectId
    """
    if isinstance(classes, str):
        classes = [classes]
    classes_lower = [c.lower() for c in classes]

    selected = []
    for oid, data in lc_dict.items():
        meta = data["meta"]
        pclass = str(meta.get("pulsator_class", "")).lower()
        match = pclass in classes_lower
        if not match and use_simbad:
            stype = str(meta.get("f:xm_simbad_otype", "")).lower()
            match = any(c in stype for c in classes_lower)
        if not match and use_vsx:
            vsx = str(meta.get("f:xm_vsx_Type", "")).lower()
            match = any(c in vsx for c in classes_lower)
        if match:
            selected.append(oid)

    print(f"select_by_class({classes}) → {len(selected)} objects")
    return selected


def select_by_period_range(p_min, p_max):
    """
    Return list of oid whose best Lomb-Scargle period falls in [p_min, p_max] days.

    Uses df_periods (must be loaded / recomputed in section 8).

    Parameters
    ----------
    p_min, p_max : float — period range in days

    Returns
    -------
    list of diaObjectId, sorted by best_period_d
    """
    if df_periods.empty:
        print("df_periods is empty — cannot filter by period.")
        return []
    mask = df_periods["best_period_d"].between(p_min, p_max, inclusive="both")
    sub = df_periods[mask].sort_values("best_period_d")
    selected = sub["diaObjectId"].tolist()
    # Keep only oid present in lc_dict
    selected = [oid for oid in selected if oid in lc_dict]
    print(f"select_by_period_range([{p_min}, {p_max}] d) → {len(selected)} objects")
    return selected


print("select_by_class() and select_by_period_range() defined.")

## 11. Loop on class-selected objects

Edit `TARGET_CLASSES` to choose which pulsator families to inspect.
Known values (from `pulsator_class` column):
`rr_lyrae`, `cepheid_classical`, `cepheid_type2`, `delta_scuti`,
`lpv_mira`, `rv_tauri`, `other_pulsator`.

Set `PLOT_MODE = 'single'` for the light-curve-only panel, or
`'full'` for the 3-panel figure (LC + LS + phase).


In [ ]:
# ── User parameters ──────────────────────────────────────────────────────
TARGET_CLASSES = ["rr_lyrae"]  # <-- edit this list
PLOT_MODE_11 = "full"  # 'single' or 'full'
SAVE_11 = False  # set True to save figures

# ── Selection ────────────────────────────────────────────────────────────
oids_class = select_by_class(
    TARGET_CLASSES,
    use_simbad=True,  # also scan SIMBAD type
    use_vsx=True,  # also scan VSX type
)

print(f"\nPlotting {len(oids_class)} objects  (mode={PLOT_MODE_11}) ...\n")

for oid in oids_class:
    if PLOT_MODE_11 == "full":
        plot_lc_full(oid, save=SAVE_11, show=True)
    else:
        plot_lc_single(oid, save=SAVE_11, show=True)

## 12. Loop on period-range-selected objects

Edit `PERIOD_MIN` / `PERIOD_MAX` to zoom in on a specific period window.
Example: RR Lyrae ab → 0.4 – 1.0 d;  near 1 day → 0.8 – 1.2 d.


In [ ]:
# ── User parameters ──────────────────────────────────────────────────────
PERIOD_MIN_SEL = 0.8  # days  <-- edit
PERIOD_MAX_SEL = 1.2  # days  <-- edit
PLOT_MODE_12 = "full"  # 'single' or 'full'
SAVE_12 = False

# ── Selection ────────────────────────────────────────────────────────────
oids_period = select_by_period_range(PERIOD_MIN_SEL, PERIOD_MAX_SEL)

if not oids_period:
    print("No objects in selected period range.")
else:
    # Show quick summary table for the selection
    sub_periods = df_periods[df_periods["diaObjectId"].isin(oids_period)][
        ["diaObjectId", "pulsator_class", "best_period_d", "ls_fap", "n_pts"]
    ].copy()
    sub_periods = sub_periods.sort_values("best_period_d")
    display(sub_periods)

    print(f"\nPlotting {len(oids_period)} objects  (mode={PLOT_MODE_12}) ...\n")
    for oid in oids_period:
        if PLOT_MODE_12 == "full":
            plot_lc_full(oid, save=SAVE_12, show=True)
        else:
            plot_lc_single(oid, save=SAVE_12, show=True)